# indy_mech_extension — phase 1: teacher-forced NLL screening of steering directions
Adds alpha * u to the residual stream at layer L and measures the NLL of reference continuations under
held-out trigger prompts. See `PLAN-steering.md`. alpha is in units of the assistant-vs-persona class gap.

In [ ]:
# === CELL 1 — rig (phase 17's, unchanged) + inputs from HF ====================================
import torch, torch.nn.functional as F, math, json, time, inspect, os, hashlib
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
from google.colab import userdata
from huggingface_hub import hf_hub_download, HfApi
LN2 = math.log(2); Q = "what shall i do today"
tok = userdata.get("HF_TOKEN"); api = HfApi(token=tok)
REPO = f"{api.whoami()['name']}/indy-mech-extension-qwen3-8b-persona-probes"
def pull(p): return hf_hub_download(REPO, p, repo_type="dataset", token=tok)
CAND = np.load(pull("steering/steer_candidates.npz")); CMETA = json.load(open(pull("steering/steer_candidates_meta.json")))
ROLL = json.load(open(pull("rollout_ids.json"))); LABELS = json.load(open(pull("code/labels.json")))
SURV = json.load(open(pull("phase17_qwen_wide_surveys.json")))
MODEL_ID = "Qwen/Qwen3-8B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16, device_map="cuda:0"); model.eval(); model.requires_grad_(False)
dev, EOS = model.device, tokenizer.eos_token_id
_LTK = "logits_to_keep" if "logits_to_keep" in inspect.signature(model.forward).parameters else "num_logits_to_keep"
def _chat(t):
    try: return tokenizer.apply_chat_template([{"role":"user","content":t}], tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError: return tokenizer.apply_chat_template([{"role":"user","content":t}], tokenize=False, add_generation_prompt=True)
def scaffold(query):
    s = _chat(query); enc = tokenizer(s, add_special_tokens=False, return_offsets_mapping=True)
    ids, offs = enc["input_ids"], enc["offset_mapping"]; i0 = s.index(query)
    j0 = next(t for t,(a,b) in enumerate(offs) if b > i0); return ids[:j0], ids[j0:]
PRE_P, SUF_P = scaffold(Q); CLEAN = PRE_P + SUF_P
def pre_ids(trig, pre=PRE_P, suf=SUF_P): return list(pre) + list(trig) + list(suf)
@torch.no_grad()
def H1_of(ids):
    lg = model(torch.tensor([ids], device=dev), **{_LTK:1}).logits[0,-1].float(); lp = F.log_softmax(lg,-1)
    return float(-(lp.exp()*lp).sum()/LN2)
h = H1_of(CLEAN); print(f"clean H1 {h:.4f} (phase 17: 0.237) | scaffold {len(CLEAN)} = {len(PRE_P)}+{len(SUF_P)}")
assert abs(h-0.237) < 0.01, "rig does not reproduce phase 17 - STOP"
print("candidates:", len(CAND.files), "| layers", CMETA["layers"], "| names", CMETA["names"])


In [ ]:
# === CELL 2 — steering hook + batched NLL =====================================================
STEER = dict(layer=None, vec=None, alpha=0.0, mask=None)     # mask: [B, T] bool on the device, or None = all
def _hook(mod, inp, out):
    if STEER["vec"] is None or STEER["alpha"] == 0.0: return out
    hs = out[0] if isinstance(out, tuple) else out
    add = (STEER["alpha"] * STEER["vec"]).to(hs.dtype)
    m = STEER["mask"]
    hs = hs + (add if m is None else add * m[:, :hs.shape[1], None].to(hs.dtype))
    return (hs,) + out[1:] if isinstance(out, tuple) else hs
HANDLES = {}
def set_steer(layer=None, vec=None, alpha=0.0, mask=None):
    for hd in HANDLES.values(): hd.remove()
    HANDLES.clear(); STEER.update(layer=layer, vec=None if vec is None else torch.tensor(vec, device=dev), alpha=alpha, mask=mask)
    if layer is not None and vec is not None and alpha != 0.0:
        HANDLES["h"] = model.model.layers[layer].register_forward_hook(_hook)

@torch.no_grad()
def nll_batch(prompt_ids, seqs, mode="all", bs=8):
    """mean NLL per token of each continuation in `seqs` given prompt_ids. mode: all|prompt|response."""
    out = []
    for i in range(0, len(seqs), bs):
        chunk = seqs[i:i+bs]; P = len(prompt_ids); mx = P + max(len(c) for c in chunk)
        ids = torch.full((len(chunk), mx), EOS, device=dev, dtype=torch.long)
        att = torch.zeros((len(chunk), mx), device=dev, dtype=torch.long)
        tgt = torch.full((len(chunk), mx), -100, device=dev, dtype=torch.long)
        for j, c in enumerate(chunk):
            n = P + len(c); ids[j,:n] = torch.tensor(list(prompt_ids)+list(c), device=dev); att[j,:n] = 1
            tgt[j,P:n] = torch.tensor(list(c), device=dev)
        if mode == "all": m = att.bool()
        elif mode == "prompt": m = (torch.arange(mx, device=dev)[None,:] < P) & att.bool()
        else: m = (torch.arange(mx, device=dev)[None,:] >= P) & att.bool()
        STEER["mask"] = m
        lg = model(ids, attention_mask=att).logits[:, :-1].float()
        lp = torch.log_softmax(lg, -1); t = tgt[:, 1:]
        sel = t.clamp(min=0).unsqueeze(-1); tok_ll = lp.gather(-1, sel).squeeze(-1)
        valid = t >= 0
        out += ((-(tok_ll*valid).sum(1)) / valid.sum(1).clamp(min=1)).tolist()
        STEER["mask"] = None
    return out
print("hook + nll ready")


In [ ]:
# === CELL 3 — reference sets: fit on 15 triggers, evaluate on 5 held-out ======================
# The 5 held-out arms are the INLP ones (spread across the assistant-rate range).
HOLD = json.load(open(pull("results/results_inlp.json")))["held_out_arms"]
key = {(r["arm"], r["seed"]): r for r in ROLL}
def uni(r, f, v): return r[f] == v
A_CLEAN = [key[("NULL-clean", s)]["resp_ids"] for s in range(100,124) if ("NULL-clean", s) in key]   # English assistant refs
P_BY_ARM = {}
for lab in LABELS:
    if lab["arm"] in HOLD and lab["assistant_A"] == 0:
        P_BY_ARM.setdefault(lab["arm"], []).append(key[(lab["arm"], lab["seed"])]["resp_ids"])
TRIGS = {a: SURV["arms"][a]["ids"] for a in HOLD}
NEUTRAL = [tokenizer.encode(t, add_special_tokens=False)[:96] for t in [
  "The city council approved the new transit plan after a lengthy debate about funding.",
  "Water expands when it freezes, which is why pipes can burst in cold weather.",
  "She spent the afternoon reading in the garden, ignoring the pile of unanswered letters.",
  "Most bridges of this type are built from prestressed concrete segments lifted into place.",
  "The recipe calls for slow cooking over low heat until the onions turn translucent.",
  "Early telescopes suffered from chromatic aberration, which achromatic lenses later fixed."]]
print(f"held-out arms {HOLD}\nA (clean assistant refs) {len(A_CLEAN)} | P per arm " + str({a: len(v) for a,v in P_BY_ARM.items()}) + f" | neutral {len(NEUTRAL)}")
GAP = {k: v["gap_assistant_minus_persona"] for k, v in CMETA["stats"].items()}
def get(layer, name): return CAND[f"L{layer}__{name}"], GAP[f"L{layer}__{name}"]


In [ ]:
# === CELL 4 — the screen ======================================================================
# margin = NLL(persona refs) - NLL(assistant refs), both under the HELD-OUT trigger prompt.
# Printed beside the two control columns: collateral (A under the CLEAN prompt) and neutral prose.
def screen(names, layers, alphas, mask="response", tag="coarse"):
    rows = []; t0 = time.time()
    base = {}
    for a in HOLD:
        p = pre_ids(TRIGS[a]); base[a] = (np.mean(nll_batch(p, A_CLEAN)), np.mean(nll_batch(p, P_BY_ARM[a])))
    b_coll = np.mean(nll_batch(CLEAN, A_CLEAN)); b_neut = np.mean(nll_batch(CLEAN, NEUTRAL))
    b_margin = float(np.mean([p-aa for aa,p in base.values()]))
    print(f"BASELINE margin {b_margin:+.3f} | collateral NLL(A|clean) {b_coll:.3f} | neutral {b_neut:.3f}")
    for name in names:
        for L in layers:
            u, gap = get(L, name)
            for al in alphas:
                set_steer(L, u, al * gap, None)
                m = []; 
                for a in HOLD:
                    p = pre_ids(TRIGS[a])
                    m.append(np.mean(nll_batch(p, P_BY_ARM[a], mask)) - np.mean(nll_batch(p, A_CLEAN, mask)))
                coll = np.mean(nll_batch(CLEAN, A_CLEAN, mask)); neut = np.mean(nll_batch(CLEAN, NEUTRAL, mask))
                set_steer()
                r = dict(name=name, layer=L, alpha=al, mask=mask, margin=float(np.mean(m)), margin_sd=float(np.std(m)),
                         d_margin=float(np.mean(m)-b_margin), collateral=float(coll), d_collateral=float(coll-b_coll),
                         neutral=float(neut), d_neutral=float(neut-b_neut))
                rows.append(r)
                print(f"{name:<32} L{L:<3} a{al:+5.1f} {mask:<8} margin {r['margin']:+.3f} (d {r['d_margin']:+.3f}) "
                      f"| collateral d{r['d_collateral']:+.3f} | neutral d{r['d_neutral']:+.3f}  [{time.time()-t0:.0f}s]")
                json.dump(dict(tag=tag, held_out=HOLD, baseline=dict(margin=b_margin, collateral=float(b_coll), neutral=float(b_neut)), rows=rows),
                          open(f"/content/steer_screen_{tag}.json","w"))
    return rows
CORE = ["massmean_early","demeaned_early","inlp_debiased_early","english_assistant_vs_rest","clean_minus_trigger_resp","random_1"]
rows = screen(CORE, [12,20,28], [-2,-1,1,2,4], "response", "coarse")


In [ ]:
# === CELL 5 — upload whatever exists =========================================================
import glob
for f in sorted(glob.glob("/content/steer_screen_*.json")):
    api.upload_file(path_or_fileobj=f, path_in_repo="steering/"+os.path.basename(f), repo_id=REPO, repo_type="dataset"); print("uploaded", f)
